In [ ]:
######### Package Imports #########################################################################
import os, warnings, copy, uuid
# remove warnings from ax
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings('ignore')
warnings.filterwarnings(action='ignore', category=FutureWarning)
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import combinations
from botorch.acquisition.multi_objective.logei import qLogExpectedHypervolumeImprovement
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import ax,torch
from ax.modelbridge.transforms.standardize_y import StandardizeY
from ax.modelbridge.transforms.unit_x import UnitX
from ax.modelbridge.transforms.remove_fixed import RemoveFixed
from ax.modelbridge.transforms.log import Log
from ax.modelbridge.transforms.one_hot import OneHot
from ax.core.base_trial import TrialStatus as T
from ax.utils.notebook.plotting import init_notebook_plotting, render
from ax.plot.slice import plot_slice

from optimpv import *
from optimpv.axBOtorch.axBOtorchOptimizer import axBOtorchOptimizer
from optimpv.TransferMatrix.TransferMatrixAgent import TransferMatrixAgent
from optimpv.TransferMatrix.TransferMatrixModel import *

init_notebook_plotting()
# warnings.filterwarnings('ignore') 
##############################################################################################
# Define the parameters to be fitted
params = []
d_3 = FitParam(name = 'd_3', value = 80e-9, bounds = [40e-9, 200e-9], log_scale = False, rescale = True,  stepsize=1e-9,  value_type = 'int', type='range', display_name='d_3',unit='m')
params.append(d_3)

# d_4 = FitParam(name = 'd_4', value = 9e-9, bounds = [5e-9, 20e-9], log_scale = False, rescale = True, value_type = 'int', stepsize=1e-9, type='range', display_name='d_4',unit='m')
# params.append(d_4)

# d_5 = FitParam(name = 'd_5', value =  100e-9, bounds = [50e-9, 200e-9], log_scale = False, rescale = True, stepsize=1e-9, value_type = 'int', type='range', display_name='d_5',unit='m')
# params.append(d_5)

d_6 = FitParam(name = 'd_6', value =  10e-9, bounds = [5e-9, 20e-9], log_scale = False, rescale = True, stepsize=1e-9, value_type = 'int', type='range', display_name='d_6',unit='m')
params.append(d_6)

d_7 = FitParam(name = 'd_7', value =  100e-9, bounds = [50e-9, 200e-9], log_scale = False, rescale = True, stepsize=1e-9, value_type = 'int', type='range', display_name='d_7',unit='m')
params.append(d_7)

d_8 = FitParam(name = 'd_8', value =  10e-9, bounds = [5e-9, 20e-9], log_scale = False, rescale = True, stepsize=1e-9, value_type = 'int', type='range', display_name='d_8',unit='m')
params.append(d_8)

d_9 = FitParam(name = 'd_9', value =  100e-9, bounds = [50e-9, 200e-9], log_scale = False, rescale = True, stepsize=1e-9, value_type = 'int', type='range', display_name='d_9',unit='m')
params.append(d_9)

nk_3 = FitParam(name = 'nk_3', value =  'PCE10_FOIC_1to1', values = ['PCE10_FOIC_1to1','P3HTPCBM_BHJ','PM6Y6Brabec'], log_scale = False, rescale = False, value_type = 'str', type='choice', display_name='nk_3',unit='')
params.append(nk_3)


# original values
params_orig = copy.deepcopy(params)
dum_dic = {}
for i in range(len(params)):
    if params[i].type == 'range':
        # if params[i].value_type == 'int':
        #     dum_dic[params[i].name] = params[i].value/params[i].stepsize
        # else:
        dum_dic[params[i].name] = params[i].value/params[i].fscale 
    elif params[i].type == 'choice':
        dum_dic[params[i].name] = params[i].value
    # dum_dic[params[i].name] = params[i].value/params[i].fscale # we need this just to run the model to generate some fake data

In [ ]:
# Initialize the agent and default device stack
layers = ['SiOx' , 'ITO' , 'ZnO' , 'PCE10_FOIC_1to1' , 'MoOx' , 'Ag', 'MoOx', 'LiF','MoOx', 'LiF','Air'] # list of layers (need to be the same than the name nk_*.csv file in the matdata folder)
thicknesses =  [0 , 100e-9 , 30e-9  , 100e-9 , 9e-9 , 8e-9, 100e-9, 100e-9, 100e-9, 100e-9, 100e-9]# list of thicknesses in nm
mat_dir = os.path.join(os.getcwd(),'Data','matdata') # path to the folder containing the nk_*.csv files
lambda_min = 350e-9 # start of the wavelength range
lambda_max = 800e-9 # end of the wavelength range
lambda_step = 1e-9 # wavelength step
x_step = 1e-9 # x step
activeLayer = 3 # active layer index
spectrum = os.path.join(mat_dir ,'AM15G.txt') # path to the AM15G spectrum file
photopic_file = os.path.join(mat_dir ,'photopic_curve.txt') # path to the photopic spectrum file

In [ ]:
print(TMM(dum_dic, layers, thicknesses, lambda_min, lambda_max, lambda_step, x_step, activeLayer, spectrum, mat_dir,photopic_file))

In [ ]:
TMAgent = TransferMatrixAgent(params, [None,None], layers=layers, thicknesses=thicknesses, lambda_min=lambda_min, lambda_max=lambda_max, lambda_step=lambda_step, x_step=x_step, activeLayer=activeLayer, spectrum=spectrum, mat_dir=mat_dir, photopic_file=photopic_file, exp_format=['Jsc', 'AVT'],metric=[None,None],loss=[None,None],threshold=[4,0.1],minimize=[False,False])

# if you want to target a specific AVT value, you can use the following line
# TMAgent = TransferMatrixAgent(params, [None,0.4], layers=layers, thicknesses=thicknesses, lambda_min=lambda_min, lambda_max=lambda_max, lambda_step=lambda_step, x_step=x_step, activeLayer=activeLayer, spectrum=spectrum, mat_dir=mat_dir, photopic_file=photopic_file, exp_format=['Jsc', 'AVT'],metric=[None,'MAE'],loss=[None,'linear'],threshold=[4,0.1],minimize=[False,False])

In [ ]:
# Define the model kwargs
from ax.models.torch.botorch_modular.utils import ModelConfig
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec
from gpytorch.kernels import MaternKernel
from gpytorch.kernels import RBFKernel
from gpytorch.constraints import Interval
from ax.modelbridge.transforms.standardize_y import StandardizeY
from ax.modelbridge.transforms.unit_x import UnitX
from ax.modelbridge.transforms.remove_fixed import RemoveFixed
from ax.modelbridge.transforms.log import Log
from ax.models.torch.botorch_modular.surrogate import Surrogate
#import single task GP
from botorch.models.gp_regression import SingleTaskGP
from optimpv.axBOtorch.EGBO import EGBOAcquisition
# model_kwargs_list = [{},{'torch_device': torch.device("cuda" if torch.cuda.is_available() else "cpu"),'torch_dtype': torch.double,'botorch_acqf_class':qLogExpectedHypervolumeImprovement}]
model_kwargs_list = [{},{'torch_device': torch.device("cuda" if torch.cuda.is_available() else "cpu"),'torch_dtype': torch.double,'botorch_acqf_class':qLogNoisyExpectedImprovement,
        'acquisition_class':EGBOAcquisition}]#,'surrogate_spec':SurrogateSpec(model_configs=[ModelConfig(botorch_model_class=SingleTaskGP,covar_module_class=MaternKernel,covar_module_options={'lengthscale_constraint':Interval(1e-6,100)})])}]
# Define the optimizer
optimizer = axBOtorchOptimizer(params = params, agents = TMAgent, models = ['SOBOL','BOTORCH_MODULAR'],n_batches = [1,20], batch_size = [10,4], ax_client = None,  max_parallelism = -1, model_kwargs_list = model_kwargs_list, model_gen_kwargs_list = None, name = 'ax_opti')

In [ ]:
optimizer.optimize(True) 

In [ ]:
ax_client = optimizer.ax_client
pareto = ax_client.get_pareto_optimal_parameters(use_model_predictions=False)
best_keys = list(pareto.keys())
print(pareto)
plt.figure()
plt.plot(ax_client.get_trace())

#find max in get_trace
max_val = np.max(ax_client.get_trace())
index_max = np.argmax(ax_client.get_trace())
index_max = best_keys[0]
best_parameters = pareto[index_max][0]

TMAgent.params_w(best_parameters,TMAgent.params)
for i in range(len(params)):
    best_parameters[params[i].name] = params[i].value

In [ ]:
# plot the evolution of the optimization
render(ax_client.get_contour_plot(param_x="d_6", param_y="d_7", metric_name=optimizer.all_metrics[0]))

model = ax_client.generation_strategy.model

render(plot_slice(model=model, param_name="d_7", metric_name=optimizer.all_metrics[0]))


In [ ]:
data = ax_client.experiment.fetch_data()

# split df by metric name 
data = data.df
metric1_df = data[data["metric_name"] == optimizer.all_metrics[0]]
metric2_df = data[data["metric_name"] == optimizer.all_metrics[1]]

# reset index
metric1_df = metric1_df.reset_index(drop=True)
metric2_df = metric2_df.reset_index(drop=True)

plt.figure()
plt.plot(np.maximum.accumulate(metric1_df["mean"]), label="Best value seen so far")
plt.xlabel('Number of iterations')
plt.ylabel('log of '+optimizer.all_metrics[0])

plt.figure()
plt.plot(np.maximum.accumulate(metric2_df["mean"]), label="Best value seen so far")
plt.xlabel('Number of iterations')
plt.ylabel('log of '+optimizer.all_metrics[1])

plt.show()


In [ ]:
from optimpv.posterior.posterior import *
params_orig_dict = {}
for p in params_orig:
    params_orig_dict[p.name] = p.value
fig_dens, ax_dens = plot_density_exploration(params, optimizer = optimizer, best_parameters = best_parameters, params_orig = params_orig_dict, optimizer_type = 'ax')

In [ ]:
comb = list(combinations(optimizer.all_metrics, 2))

import matplotlib
cm = matplotlib.colormaps.get_cmap('viridis')
df = get_df_from_ax(params, optimizer)
# create pareto df
dum_dic = {}
for eto in pareto.keys():
    for metr in optimizer.all_metrics:
        if metr not in dum_dic.keys():
            dum_dic[metr] = []
        dum_dic[metr].append(pareto[eto][1][0][metr])
df_pareto = pd.DataFrame(dum_dic)

for c in comb:
    plt.figure()
    plt.scatter(df[c[0]],df[c[1]],c=df['iteration'],cmap=cm)
    sorted_df = df_pareto.sort_values(by=c[0])
    plt.plot(sorted_df[c[0]],sorted_df[c[1]],'r')
    
    plt.xlabel(c[0].split('_')[1])
    plt.ylabel(c[1].split('_')[1])
    cbar = plt.colorbar()
    cbar.set_label('Iteration')
    plt.show()



In [ ]:
from botorch.utils.multi_objective.hypervolume import Hypervolume
# make a tensor with TMAgent.threshold
threshold = torch.tensor(TMAgent.threshold, device=model.device)

# calculate HV
hv = np.zeros(len(df))
for i in range(len(df)):
    y = torch.tensor(df[optimizer.all_metrics].values[:i+1], device=model.device)
    hv[i] = Hypervolume(ref_point=threshold).compute(y)

plt.figure()
plt.plot(hv)
plt.xlabel('Iteration')

In [ ]:

for c in comb:
    plt.figure()
    # density plot
    sns.kdeplot(data=df, x=c[0], y=c[1], fill=True, zorder=0, levels=100, cmap='Blues')
    # # color white anything above sorted_df
    plt.fill_between(sorted_df[c[0]], sorted_df[c[1]], max(sorted_df[c[1]].max(),df[c[1]].max()), color='white')
    plt.scatter(df[c[0]],df[c[1]],c='k',s=10)
    sorted_df = df_pareto.sort_values(by=c[0])
    plt.plot(sorted_df[c[0]],sorted_df[c[1]],'k')
    
    # Set y-axis limits to ensure shading goes all the way to the axis
    plt.ylim(bottom=0, top=sorted_df[c[1]].max())
    plt.xlim(left=0, right=sorted_df[c[0]].max())
    # plt.scatter(df[c[0]],df[c[1]],c='k')
    # cbar = plt.colorbar()
    # cbar.set_label('Iteration')
    plt.xlabel(c[0].split('_')[1])
    plt.ylabel(c[1].split('_')[1])
    
    plt.show()

